Name of Machine Leaarning Model: Complement Naive Bayes

In [1]:
# Loading the data 
from pathlib import Path

import kagglehub
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score, classification_report

COMPETITION_NAME = "50-007-machine-learning-may-2026"

competition_path = Path(kagglehub.competition_download(COMPETITION_NAME))

train_features_df = pd.read_csv(competition_path / "train_features.csv")

test_features_df = pd.read_csv(competition_path / "test_features.csv")

submission_df = pd.read_csv(competition_path / "sample_submission.csv")

id_col, label_col = submission_df.columns[0], submission_df.columns[1]

# separate features from id/label
feature_cols = []
for c in train_features_df.columns:
    if c not in ["id", "label"]:
        feature_cols.append(c)

X_all = train_features_df[feature_cols].to_numpy(dtype=float)
y_all = train_features_df[label_col].to_numpy()
X_test = test_features_df[feature_cols].to_numpy(dtype=float)

if (X_all < 0).any():
    raise ValueError(
        "Some features are negative. Complement Naives Bayes requires non-negative features."
        "Rescale or drop the columns before fitting."
        )

X_train, X_val, y_train, y_val = train_test_split(X_all, y_all, test_size=0.2, random_state=1, stratify=y_all)

C:\Users\siawx\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
import os

# fit a complement naive bayes model
# alpha: additive smoothing parameter
# norm: whether to apply cnb weight-normalization step
# returns model: dict w/ keys 'classes', 'log_weights', 'alpha', 'norm'
def cnb_fit(X, y, alpha: float = 1.0, norm: bool=True) -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y)

    classes, y_idx = np.unique(y, return_inverse=True)
    n_classes = len(classes)
    n_features = X.shape[1]

    # feature_count_per_class[c, i]: total count of feature i in class c's docs
    feature_count_per_class = np.zeros((n_classes, n_features))
    for c in range(n_classes):
        class_mask = (y_idx == c)
        feature_count_per_class[c, :] = X[class_mask].sum(axis=0)

    total_count_per_class = feature_count_per_class.sum(axis=1) # shape (n_classes,)

    # complement_count[c]: sum over all classes != c
    complement_feature_count = (feature_count_per_class.sum(axis=0, keepdims=True) - feature_count_per_class)
    complement_total_count = total_count_per_class.sum() - total_count_per_class

    numerator = alpha + complement_feature_count
    denominator = (alpha * n_features) + complement_total_count[:, None]
    theta = numerator / denominator

    log_weights = np.log(theta) # w_ci, shape (n_classes, n_features)

    if norm:
        log_weights = log_weights / np.abs(log_weights).sum(axis=1, keepdims=True)

    return {
        "classes": classes,
        "log_weights": log_weights,
        "alpha": alpha,
        "norm": norm
    }

def cnb_decision_scores(model: dict, X) -> np.ndarray:
    # lower score -> more likely that class (CNB convention)
    X = np.asarray(X, dtype=float)
    return X @ model["log_weights"].T   # shape (n_samples, n_classes)

def cnb_predict(model: dict, X) -> np.ndarray:
    scores = cnb_decision_scores(model, X)
    best_idx = np.argmin(scores, axis=1)
    return model["classes"][best_idx]

# softmax-converted confidence scores (not a true CNB probability but useful for threshold tuning to optimise F1 directly)
def cnb_predict_proba(model: dict, X) -> np.ndarray:
    scores = cnb_decision_scores(model, X)
    neg_scores = -scores    # now higher -> more likely; w/o this flip, applying softmax directly will give a confident-looking probability for the wrong class
    neg_scores -= neg_scores.max(axis=1, keepdims=True) # numerical stability; w/o it, np.exp() -> infinity
    exp_scores = np.exp(neg_scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)

In [3]:
# stratified k-fold cross-validation
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1)

# hyperparameter sweep on alpha & norm
best_alpha, best_norm, best_f1, best_model = None, None, -1, None
for alpha in [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 8.0, 10.0, 12.0, 15.0, 20.0]:
    for norm in [True, False]:
            fold_f1_scores = []

            for train_idx, val_idx in skf.split(X_all, y_all):
                X_tr_fold, X_val_fold = X_all[train_idx], X_all[val_idx]
                y_tr_fold, y_val_fold = y_all[train_idx], y_all[val_idx]

                fold_model = cnb_fit(X_tr_fold, y_tr_fold, alpha=alpha, norm=norm)
                fold_pred = cnb_predict(fold_model, X_val_fold)
                fold_f1_scores.append(f1_score(y_val_fold, fold_pred, average='macro'))

            f1 = np.mean(fold_f1_scores)
            print(f"alpha={alpha} norm={norm} val F1={f1:.4f}")

            if f1 > best_f1:
                model = cnb_fit(X_train, y_train, alpha=alpha, norm=norm)
                best_alpha, best_norm, best_f1, best_model = alpha, norm, f1, model

print(f"\nBest alpha: {best_alpha}, best norm: {best_norm} (val F1={best_f1:.4f})")
print(classification_report(y_val, cnb_predict(best_model, X_val)))

alpha=0.01 norm=True val F1=0.6453
alpha=0.01 norm=False val F1=0.6562
alpha=0.05 norm=True val F1=0.6452
alpha=0.05 norm=False val F1=0.6559
alpha=0.1 norm=True val F1=0.6464
alpha=0.1 norm=False val F1=0.6557
alpha=0.5 norm=True val F1=0.6578
alpha=0.5 norm=False val F1=0.6536
alpha=1.0 norm=True val F1=0.6668
alpha=1.0 norm=False val F1=0.6533
alpha=2.0 norm=True val F1=0.6708
alpha=2.0 norm=False val F1=0.6560
alpha=5.0 norm=True val F1=0.6530
alpha=5.0 norm=False val F1=0.6690
alpha=8.0 norm=True val F1=0.6187
alpha=8.0 norm=False val F1=0.6718
alpha=10.0 norm=True val F1=0.6028
alpha=10.0 norm=False val F1=0.6649
alpha=12.0 norm=True val F1=0.5901
alpha=12.0 norm=False val F1=0.6565
alpha=15.0 norm=True val F1=0.5735
alpha=15.0 norm=False val F1=0.6415
alpha=20.0 norm=True val F1=0.5541
alpha=20.0 norm=False val F1=0.6155

Best alpha: 8.0, best norm: False (val F1=0.6718)
              precision    recall  f1-score   support

           0       0.58      0.58      0.58      1499


In [4]:
# fit on full training data with best alpha/norm, predict test set
final_model = cnb_fit(X_all, y_all, alpha=best_alpha, norm=best_norm)
test_pred = cnb_predict(final_model, X_test)

In [ ]:
# Build submission
submission_df["label"] = test_pred

# make sure outputs directory exist
os.makedirs("outputs", exist_ok = True)

submission_df.to_csv("outputs/submission_cnb.csv", index=False)
print("Saved predictions to submission.csv")

print("\nPredicted class distribution:")
print(pd.Series(test_pred).value_counts())

Saved predictions to submission.csv

Predicted class distribution:
1    5577
0    1422
Name: count, dtype: int64
